# Vierteljährliche Pro-forma-Gewinn-und-Verlust-Rechnung mit PROC COMPUTAB

## Zusammenfassung

Dieses Notebook erstellt die vierteljährliche Pro-forma-Gewinn-und-Verlust-Rechnung einer Regionalbank direkt aus monatlichen Buchungsdaten mit PROC COMPUTAB, der Tabellen-Berichtsprozedur von SAS/ETS. Es leitet die Zinserträge, Zinsaufwendungen, Provisionserträge und Betriebskosten jedes Monats in die richtige Kalenderquartalsspalte, verwendet dann Zeilenprogrammierblöcke, um den Nettozinsertrag, das Vorsteuerergebnis und den Jahresüberschuss zu berechnen, sowie einen Spaltenblock, um die vier Quartale zu einer Gesamtjahressumme aufzurollen.

## Datenquellen

Die einzelne synthetische Datenmenge `bank_ledger` simuliert ein Geschäftsjahr monatlicher Abschlussposten für eine mittelgroße Regionalbank. Zwölf monatliche Beobachtungen werden inline mit `call streaminit`/`rand` erzeugt, sodass das Notebook vollständig eigenständig ist.

| Variable | Typ | Beschreibung |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Abschlussstichtag zum Monatsende (Jan–Dez) |
| `loanint`  | Num | Zinsen und Gebühren aus dem Kreditportfolio (Tsd. USD) |
| `secint`   | Num | Zinserträge aus dem Wertpapierbestand (Tsd. USD) |
| `depint`   | Num | Auf Einlagen gezahlte Zinsen (Tsd. USD) |
| `borrint`  | Num | Auf Kreditaufnahmen / FHLB-Vorschüsse gezahlte Zinsen (Tsd. USD) |
| `feeinc`   | Num | Zinsunabhängige Erträge (Gebühren & Serviceentgelte) (Tsd. USD) |
| `salaries` | Num | Aufwand für Gehälter und Sozialleistungen (Tsd. USD) |
| `occupancy`| Num | Aufwand für Raum und Ausstattung (Tsd. USD) |
| `othropex` | Num | Sonstiger zinsunabhängiger Betriebsaufwand (Tsd. USD) |
| `provision`| Num | Risikovorsorge für Kreditausfälle (Tsd. USD) |
| `taxrate`  | Num | Effektiver Steuersatz auf das Vorsteuerergebnis |

# Vierteljährliche Pro-forma-Gewinn-und-Verlust-Rechnung mit PROC COMPUTAB

Finanzteams von Banken verwandeln routinemäßig ein monatliches Hauptbuch in eine **vierteljährliche Gewinn-und-Verlust-Rechnung**, die den Nettozinsertrag und den Jahresüberschuss als Endergebnis zeigt. `PROC COMPUTAB` (SAS/ETS) ist genau dafür gemacht: Es legt eine programmierbare Tabelle an, deren **Spalten** die Berichtsperioden und deren **Zeilen** die Abschlussposten sind, und es erlaubt, Zwischensummen mit gewöhnlicher DATA-Step-Logik in Zeilen- und Spaltenblöcken zu berechnen.

In diesem Notebook:

1. erzeugen wir ein Geschäftsjahr synthetischer monatlicher Buchungsdaten für eine Regionalbank.
2. leiten wir jeden Monat in sein Kalenderquartal und erstellen eine vierteljährliche Gewinn-und-Verlust-Rechnung.
3. berechnen wir Nettozinsertrag, Vorsteuerergebnis und Jahresüberschuss in einem **Zeilenblock** und rollen die Quartale in einem **Spaltenblock** zu einer Gesamtjahressumme auf.
4. verwenden wir die `out=`-Tabelle erneut, damit die berechnete Rechnung nachgelagerte Berichte speisen kann.

## Schritt 1 — Erzeugung synthetischer monatlicher Buchungsdaten

Wir simulieren zwölf Beobachtungen zum Monatsende. Die Zinserträge aus Krediten und Wertpapieren steigen im Jahresverlauf leicht an, die Einlagen- und Refinanzierungskosten skalieren mit dem Zinsumfeld, und Provisionserträge, Betriebsaufwendungen und die Risikovorsorge für Kreditausfälle tragen ein realistisches Rauschen von Monat zu Monat. `call streaminit` legt den Seed fest, sodass die Daten reproduzierbar sind.

In [1]:
DATEN bank_ledger;
   AUFRUFEN streaminit(20240115);
   format stmtdate date9.;
   AUSFÜHRUNG month = 1 BIS 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      AUSGABE;
   ENDE;
   BEHALTEN stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=bank_ledger noobs BEZEICHNUNG;
   TITEL 'Synthetic Monthly Ledger — Community Bank FY2024';
   format loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
AUSFÜHREN;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## Schritt 2 — Erstellung der vierteljährlichen Gewinn-und-Verlust-Rechnung

Das Herzstück des Berichts ist der `PROC COMPUTAB`-Step unten.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definiert vier Quartalsspalten plus eine Gesamtjahresspalte.
- Der unbeschriftete **Eingabeblock** setzt die automatische Variable **`_col_`** mit `qtr(stmtdate)`, das jede monatliche Beobachtung in die richtige Quartalsspalte leitet. Da die Eingabe standardmäßig transponiert wird, landet jede Buchungsvariable in der Zeile, die ihren Namen teilt.
- Der **Zeilenblock** `inc_stmt:` läuft einmal je Spalte und berechnet die abgeleiteten Zeilen — Nettozinsertrag, gesamten zinsunabhängigen Aufwand, Vorsteuerergebnis und Jahresüberschuss — genau so, wie es ein Buchhalter täte.
- Der **Spaltenblock** `total:` läuft einmal je Zeile und summiert die vier Quartale in die Spalte `fy` (Gesamtjahr).

Die `rows ... / ...`-Anweisungen fügen Titel, Einrückungen und Linien hinzu (`ol` Oberlinie, `ul` Unterlinie, `dul` Doppelunterlinie), sodass die Ausgabe wie ein echter Jahresabschluss aussieht.

In [2]:
TITEL 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

PROZEDUR computab DATEN=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / format=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
AUSFÜHREN;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## Schritt 3 — Wiederverwendung der COMPUTAB-Ausgabedatenmenge

Der `PROC COMPUTAB`-Step oben schrieb seine berechnete Tabelle über `out=` nach `qtr_income`. Jede Zeile dieser Datenmenge ist ein Abschlussposten und jede Spaltenvariable (`qtr1`–`qtr4`, `fy`) enthält den berechneten Wert, sodass sie nachgelagerte Berichte speisen kann. Unten geben wir die aufgerollte Gesamtjahresspalte aus, um zu bestätigen, dass die Zahlen aufgehen.

In [3]:
TITEL 'COMPUTAB Output Dataset — Full-Year Column';

PROZEDUR DRUCKEN DATEN=qtr_income BEZEICHNUNG noobs;
   VAR _row_ fy;
   format fy comma12.1;
   BEZEICHNUNG _row_ = 'Line Item' fy = 'Full Year';
AUSFÜHREN;

TITEL;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interpretation der Ergebnisse

Die Pro-forma-Rechnung liest sich von oben nach unten wie der aufsichtsrechtliche Call Report einer Bank: Der gesamte Zinsertrag abzüglich des gesamten Zinsaufwands ergibt den **Nettozinsertrag** — hier etwa \$20.5 Millionen für das Jahr — den primären Ertragstreiber der Bank. Zieht man die **Risikovorsorge für Kreditausfälle** ab, addiert die **Provisionserträge** und rechnet den **zinsunabhängigen Aufwand** heraus, ergibt sich ein Vorsteuerergebnis von rund \$9.0 Millionen, und die Anwendung des effektiven Steuersatzes von 21% ergibt einen **Jahresüberschuss** von nahezu \$7.1 Millionen. Der Spaltenblock `total:` addiert die vier Quartale in die Gesamtjahresspalte, sodass sich die Jahressummen konstruktionsbedingt mit der Summe der Quartale abstimmen — bestätigt in der `out=`-Datenmenge, wo der Gesamtjahres-`netinc` von 7,081.6 der Summe der vier Quartalswerte entspricht.

Da alles innerhalb der programmierbaren Zeilen- und Spaltenblöcke von `PROC COMPUTAB` berechnet wird, erfordert das Einsetzen eines echten monatlichen Hauptbuchs keine Änderung der Berichtslogik — nur die Eingabedatenmenge ändert sich. Die `out=`-Datenmenge macht die berechnete Rechnung dann für Dashboards, Trendanalysen oder die Automatisierung von Vorstandsunterlagen verfügbar.